In [3]:
%pwd

'D:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model'

# 1. Import the libraries:

In [4]:
import os
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                               GradientBoostingClassifier)
from xgboost import XGBClassifier
from sklearn.svm import SVC
from scipy.stats import loguniform, uniform
import matplotlib.pyplot as plt
import numpy as np
import time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib
import dagshub
from dotenv import load_dotenv
# os.chdir("..")

In [5]:
load_dotenv()

True

In [6]:
%pwd

'D:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model'

# 2. Configuration:

In [7]:
mlflow_tracking_uri = os.getenv("MLFLOW_URI")
dagshub_owner_name = os.getenv("REPO_OWNER")
dagshub_repo_name = os.getenv("REPO_NAME")
dagshub_user_token = os.getenv("DAGSHUB_USER_TOKEN")

In [8]:
mlflow.set_tracking_uri(mlflow_tracking_uri)
dagshub.init(repo_owner=dagshub_owner_name, repo_name=dagshub_repo_name, mlflow=True)

Accessing as Nishan-k

Initialized MLflow to track repo "Nishan-k/An-end-to-end-customer-churn-prediction-model"

Repository Nishan-k/An-end-to-end-customer-churn-prediction-model initialized!

# 3. Common functions:

In [9]:
def create_dep_indep_features(df:pd.DataFrame):
    """
    Splits the data into dependent(y) and independent(X) features:
    """
    df.dropna(inplace=True)
    X = df.drop(columns=['customerid', 'churn'])
    y = df['churn']
    return X, y   

In [10]:
def separate_col_by_dtypes(df:pd.DataFrame) -> list():
    """
    Returns the numerical and categorical feature names as list:
    """
    numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = df.select_dtypes(include=['object', 'string']).columns.tolist()
    return (numerical_features, categorical_features)

In [11]:
def log_confusion_matrix(y_true, y_pred,
                          model_name: str, branch: str):
    fig, ax = plt.subplots(figsize=(6, 5))
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Retained', 'Churned']
    )
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{model_name} ({branch}) — Confusion Matrix')
    plt.tight_layout()
    path = f"cm_{model_name}_{branch}.png"
    plt.savefig(path)
    plt.close()
    mlflow.log_artifact(path)
    os.remove(path)

# 4. Read the data:

In [12]:
clipped_train = pd.read_parquet("Artifacts/feature_engineering/data_clipped/train.parquet")
clipped_test = pd.read_parquet("Artifacts/feature_engineering/data_clipped/test.parquet")

unclipped_train = pd.read_parquet("Artifacts/feature_engineering/data_unclipped/train.parquet")
unclipped_test = pd.read_parquet("Artifacts/feature_engineering/data_unclipped/test.parquet")

## 4.1 Split the data into train and test sets:

In [13]:
# Clipped data:
X_train_clipped, y_train_clipped = create_dep_indep_features(clipped_train)
X_test_clipped, y_test_clipped = create_dep_indep_features(clipped_test)

# Unclipped data:
X_train_unclipped, y_train_unclipped = create_dep_indep_features(unclipped_train)
X_test_unclipped, y_test_unclipped = create_dep_indep_features(unclipped_test)

In [14]:
# The column names will be the same for both the brances(clipped & Unclipped):
numerical_features, categorical_features = separate_col_by_dtypes(X_train_clipped)

## 4.2 Branch Configuration:

In [15]:
branches = {
    'clipped': {
        'X_train': X_train_clipped,
        'y_train': y_train_clipped,
        'X_test': X_test_clipped,
        'y_test': y_test_clipped,
        'numerical_features': numerical_features,
        'categorical_features': categorical_features
    },

    'unclipped': {
        'X_train': X_train_unclipped,
        'y_train': y_train_unclipped,
        'X_test': X_test_unclipped,
        'y_test': y_test_unclipped,
        'numerical_features': numerical_features,
        'categorical_features': categorical_features
    }
}

# 5. Model Configurations:

#### During spot-checking 5 algorithms were picked:

In [16]:
model_configs = {
    
    'SVC': {
        'model': SVC(class_weight='balanced', random_state=42,
                     probability=True),
        'params': {
            'clf__C':      loguniform(0.01, 100),
            'clf__kernel': ['rbf', 'linear'],
            'clf__gamma':  ['scale', 'auto']
        }
    },
    
    'GradientBoosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'clf__n_estimators':     [100, 200, 300],
            'clf__learning_rate':    loguniform(0.01, 0.3),
            'clf__max_depth':        [3, 4, 5],
            'clf__subsample':        uniform(0.7, 0.3),
            'clf__min_samples_leaf': [10, 20, 30]
        }
    },
    
    'RandomForest': {
        'model': RandomForestClassifier(class_weight='balanced',
                                         random_state=42),
        'params': {
            'clf__n_estimators':     [100, 200, 300],
            'clf__max_depth':        [None, 5, 10, 20],
            'clf__min_samples_leaf': [1, 5, 10, 20],
            'clf__max_features':     ['sqrt', 'log2']
        }
    },
    
    'ExtraTrees': {
        'model': ExtraTreesClassifier(class_weight='balanced',
                                       random_state=42),
        'params': {
            'clf__n_estimators':     [100, 200, 300],
            'clf__max_depth':        [None, 5, 10, 20],
            'clf__min_samples_leaf': [1, 5, 10, 20],
            'clf__max_features':     ['sqrt', 'log2']
        }
    },
    
    'XGBoost': {
    'model': XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=42,
        verbosity=0
    ),
    'params': {
        'clf__n_estimators':     [100, 200, 300],
        'clf__max_depth':        [3, 5, 7, 10],
        'clf__learning_rate':    [0.01, 0.05, 0.1, 0.3],
        'clf__subsample':        [0.6, 0.8, 1.0],
        'clf__colsample_bytree': [0.6, 0.8, 1.0],
        'clf__min_child_weight': [1, 3, 5],
        'clf__gamma':            [0, 0.1, 0.3]
    }
}
}

# 6. Build the pre-processors and encoders:

In [21]:
def build_preprocessor(numerical_features, categorical_features):
    preprocessor = ColumnTransformer([
        ('num', RobustScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ], remainder='drop')
    return preprocessor

# 7. Run the experiments:

In [24]:
def run_experiment(model_name: str, model, param_grid:dict, 
                   branch_name:str, branch_data:dict, 
                   n_iter:int = 30):
    X_train = branch_data['X_train']
    y_train = branch_data['y_train']
    X_test = branch_data['X_test']
    y_test = branch_data['y_test']
    numerical_features = branch_data['numerical_features']
    categorical_features = branch_data['categorical_features']

    preprocessor = build_preprocessor(numerical_features, categorical_features)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('clf', model)
    ])

    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_grid,
        n_iter = n_iter,
        scoring = 'f1',
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        random_state=42,
        n_jobs=-1,
        verbose=0
    )

    run_name = f"{model_name}_{branch_name}"

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("model", model_name)
        mlflow.set_tag("branch", branch_name),
        mlflow.set_tag("n_features", str(len(numerical_features + categorical_features)))
        mlflow.set_tag("scaler", "RobustScaler")
        mlflow.set_tag("encoder", "OneHotEncoder")

        start = time.perf_counter()
        search.fit(X_train, y_train)
        duration = round((time.perf_counter() - start)/60, 3)
        best_params = {k.replace('clf__', ''): v for k, v in search.best_params_.items()}
        print(best_params)
        mlflow.log_params(best_params)
        mlflow.log_param("training_duraton_min", duration)
        mlflow.log_metric("cv_f1_best", search.best_score_)

        best_pipeline = search.best_estimator_
        y_pred = best_pipeline.predict(X_test)

        mlflow.log_metric("test_f1", f1_score(y_test, y_pred))
        mlflow.log_metric("test_precision", precision_score(y_test, y_pred))
        mlflow.log_metric("test_recall", recall_score(y_test, y_pred))
        mlflow.log_metric("test_accuracy", accuracy_score(y_test, y_pred))

        log_confusion_matrix(y_test, y_pred, model_name, branch_name)

        print(f"\n{'-'* 50}")
        print(f"Model: {model_name} | Branch: {branch_name}")
        print(f"CV F1: {search.best_score_*100:.2f}% | Duration:{duration}min")
        print(f"Best Params: {best_params}")
        print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))

        signature = infer_signature(X_train, y_pred)
        mlflow.sklearn.log_model(
            best_pipeline,
            artifact_path="model",
            signature=signature
        )

        return best_pipeline, search.best_score_, f1_score(y_test, y_pred)

In [25]:
all_results = []

for branch_name, branch_data in branches.items():
    print(f"\n{'#'*55}")
    print(f"BRANCH: {branch_name.upper()}")
    print(f"{'#'*55}")

    for model_name, config in model_configs.items():
        pipeline, cv_f1, test_f1 = run_experiment(
            model_name=model_name,
            model=config['model'],
            param_grid=config['params'],
            branch_name=branch_name,
            branch_data=branch_data
        )
        all_results.append({
            'model':    model_name,
            'branch':   branch_name,
            'cv_f1':    round(cv_f1 * 100, 2),
            'test_f1':  round(test_f1 * 100, 2)
        })


#######################################################
BRANCH: CLIPPED
#######################################################
{'C': np.float64(0.04207988669606638), 'gamma': 'scale', 'kernel': 'rbf'}


C:\Users\karki\anaconda3\envs\churn_prediction\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/03/24 22:33:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



--------------------------------------------------
Model: SVC | Branch: clipped
CV F1: 82.31% | Duration:13.96min
Best Params: {'C': np.float64(0.04207988669606638), 'gamma': 'scale', 'kernel': 'rbf'}
              precision    recall  f1-score   support

    Retained       0.76      0.26      0.39      1282
     Churned       0.74      0.96      0.84      2859

    accuracy                           0.75      4141
   macro avg       0.75      0.61      0.61      4141
weighted avg       0.75      0.75      0.70      4141



2026/03/24 22:33:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run SVC_clipped at: https://dagshub.com/Nishan-k/An-end-to-end-customer-churn-prediction-model.mlflow/#/experiments/0/runs/1a67992ea4a3468984bcdb6c60ed8214
🧪 View experiment at: https://dagshub.com/Nishan-k/An-end-to-end-customer-churn-prediction-model.mlflow/#/experiments/0
{'learning_rate': np.float64(0.010189592979395137), 'max_depth': 5, 'min_samples_leaf': 30, 'n_estimators': 100, 'subsample': np.float64(0.9134025858245949)}


...

🏃 View run ExtraTrees_unclipped at: https://dagshub.com/Nishan-k/An-end-to-end-customer-churn-prediction-model.mlflow/#/experiments/0/runs/2c2a19511d574f3198d19a615bbf4bd2
🧪 View experiment at: https://dagshub.com/Nishan-k/An-end-to-end-customer-churn-prediction-model.mlflow/#/experiments/0
{'subsample': 0.8, 'n_estimators': 100, 'min_child_weight': 3, 'max_depth': 10, 'learning_rate': 0.01, 'gamma': 0.3, 'colsample_bytree': 0.6}


C:\Users\karki\anaconda3\envs\churn_prediction\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/03/24 23:01:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



--------------------------------------------------
Model: XGBoost | Branch: unclipped
CV F1: 84.81% | Duration:0.278min
Best Params: {'subsample': 0.8, 'n_estimators': 100, 'min_child_weight': 3, 'max_depth': 10, 'learning_rate': 0.01, 'gamma': 0.3, 'colsample_bytree': 0.6}
              precision    recall  f1-score   support

    Retained       0.87      0.18      0.30      1282
     Churned       0.73      0.99      0.84      2859

    accuracy                           0.74      4141
   macro avg       0.80      0.59      0.57      4141
weighted avg       0.77      0.74      0.67      4141



2026/03/24 23:01:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_unclipped at: https://dagshub.com/Nishan-k/An-end-to-end-customer-churn-prediction-model.mlflow/#/experiments/0/runs/d1f523bb6c45459b91bf9e9d9bde2200
🧪 View experiment at: https://dagshub.com/Nishan-k/An-end-to-end-customer-churn-prediction-model.mlflow/#/experiments/0


In [26]:
results_df = pd.DataFrame(all_results).sort_values(
    'test_f1', ascending=False).reset_index(drop=True)

print("\n=== FINAL COMPARISON — All 8 Experiments ===")
print(results_df.to_string(index=False))

best = results_df.iloc[0]
print(f"\nWinner: {best['model']} on {best['branch']} data")
print(f"Test F1: {best['test_f1']}%")


=== FINAL COMPARISON — All 8 Experiments ===
           model    branch  cv_f1  test_f1
      ExtraTrees unclipped  83.81    84.92
      ExtraTrees   clipped  84.19    84.81
GradientBoosting unclipped  84.82    84.54
GradientBoosting   clipped  84.68    84.53
    RandomForest unclipped  84.40    84.49
         XGBoost   clipped  84.90    84.45
    RandomForest   clipped  84.43    84.44
             SVC   clipped  82.31    83.98
         XGBoost unclipped  84.81    83.94
             SVC unclipped  83.17    83.55

Winner: ExtraTrees on unclipped data
Test F1: 84.92%
